In [2]:
# 1. Load raw data
import json
from pathlib import Path

import pandas as pd

INPUT_DIR = Path(
    "../../data/raw/individual/ingestion_date=2026-09-27"
)

json_files = list(INPUT_DIR.glob("app_*.json"))

print(f"Found {len(json_files)} JSON files.")

Found 1000 JSON files.


In [3]:
# 2. Extract raw fields

def extract_app_data(file_path):
    with file_path.open("r", encoding="utf-8") as file:
        response = json.load(file)

    # The response is keyed by appid
    if not response:
        print(f"Skipping empty response: {file_path}")
        return None

    appid_key = next(iter(response))
    app = response[appid_key].get("data")

    if not app:
        return None

    price = app.get("price_overview") or {}
    metacritic = app.get("metacritic") or {}
    release_date = app.get("release_date") or {}
    recommendations = app.get("recommendations") or {}
    platforms = app.get("platforms") or {}

    return {
        "type": app.get("type"),
        "name": app.get("name"),
        "steam_appid": app.get("steam_appid"),
        "required_age": app.get("required_age"),
        "is_free": app.get("is_free"),

        "currency": price.get("currency"),
        "price_initial": price.get("initial"),
        "price_final": price.get("final"),
        "discount_percent": price.get("discount_percent"),

        "developers": app.get("developers") or [],
        "publishers": app.get("publishers") or [],

        "windows": platforms.get("windows"),
        "mac": platforms.get("mac"),
        "linux": platforms.get("linux"),

        "genres": app.get("genres") or [],
        "categories": app.get("categories") or [],

        "metacritic_score": metacritic.get("score"),
        "metacritic_url": metacritic.get("url"),

        "coming_soon": release_date.get("coming_soon"),
        "release_date": release_date.get("date"),

        "recommendations_total": recommendations.get("total"),
    }

In [4]:
# 3. Data quality checks and Transform scalar fields
records = []

for file_path in json_files:
    record = extract_app_data(file_path)

    if record is not None:
        records.append(record)

print(f"Extracted {len(records)} apps.")

# The first run I got 985/1000 succesfully transmited trough the API

Extracted 985 apps.


In [5]:
# Identify missing apps in order to retry the API fetch
missing_apps = []

for file_path in json_files:
    with file_path.open("r", encoding="utf-8") as file:
        response = json.load(file)

    appid_key = next(iter(response))
    app_data = response[appid_key].get("data")

    if not app_data:
        missing_apps.append(appid_key)

print(f"Missing apps: {len(missing_apps)}")
print(missing_apps)

# I retried to fetch them, no success, so 15 invalid apps remain. 
# I'll work with the rest, the 985 apps

Missing apps: 15
['11610', '12650', '12660', '15350', '19030', '19080', '19810', '22380', '33550', '33700', '33710', '33720', '35030', '39160', '45750']


In [6]:
df = pd.DataFrame(records)

print(df.shape)
print(df.head())

(985, 21)
   type                        name  steam_appid required_age  is_free  \
0  game              Counter-Strike           10            0    False   
1  game            Rag Doll Kung Fu         1002            0    False   
2  game  Call of Duty: World at War        10090           18    False   
3  game    King's Quest™ Collection        10100            0    False   
4  game     Space Quest™ Collection        10110            0    False   

  currency  price_initial  price_final  discount_percent     developers  ...  \
0      EUR          819.0        819.0               0.0        [Valve]  ...   
1      EUR           79.0         79.0               0.0  [Mark Healey]  ...   
2      EUR         1999.0       1999.0               0.0     [Treyarch]  ...   
3      EUR         1999.0       1999.0               0.0       [Sierra]  ...   
4      EUR          999.0        999.0               0.0       [Sierra]  ...   

  windows    mac  linux                                      gen

In [7]:
# data types and missing values
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 985 entries, 0 to 984
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   type                   985 non-null    str    
 1   name                   985 non-null    str    
 2   steam_appid            985 non-null    int64  
 3   required_age           985 non-null    object 
 4   is_free                985 non-null    bool   
 5   currency               925 non-null    str    
 6   price_initial          925 non-null    float64
 7   price_final            925 non-null    float64
 8   discount_percent       925 non-null    float64
 9   developers             985 non-null    object 
 10  publishers             985 non-null    object 
 11  windows                985 non-null    bool   
 12  mac                    985 non-null    bool   
 13  linux                  985 non-null    bool   
 14  genres                 985 non-null    object 
 15  categories       

type                       0
name                       0
steam_appid                0
required_age               0
is_free                    0
currency                  60
price_initial             60
price_final               60
discount_percent          60
developers                 0
publishers                 0
windows                    0
mac                        0
linux                      0
genres                     0
categories                 0
metacritic_score         411
metacritic_url           411
coming_soon                0
release_date               0
recommendations_total    206
dtype: int64

In [8]:
# Required_age is object? 
df["required_age"].value_counts(dropna=False)


required_age
0     926
18     34
16     17
17      5
15      3
Name: count, dtype: int64

In [9]:
# I'll convert it to numerical
df["required_age"] = pd.to_numeric(df["required_age"])
df["required_age"].dtype

dtype('int64')

In [10]:
numeric_columns = [
    "price_initial",
    "price_final",
    "discount_percent",
    "metacritic_score",
    "recommendations_total",
]

df[numeric_columns].describe()

# - No negative prices or discounts were found.
# - Discount percentages are within the expected 0-100 range.
# - Metacritic scores are within the expected 0-100 range.
# - Recommendation counts are positive.
# - Missing values occur only in optional fields.

,price_initial,price_final,discount_percent,metacritic_score,recommendations_total
count,925.000000,925.000000,925.000000,574.000000,7.790000e+02
mean,1008.552432,883.574054,10.558919,74.337979,1.645027e+04
std,631.177806,601.296521,26.338416,10.997257,1.968156e+05
min,79.000000,79.000000,0.000000,24.000000,1.010000e+02
25%,499.000000,499.000000,0.000000,68.000000,3.030000e+02
50%,975.000000,819.000000,0.000000,76.000000,8.950000e+02
75%,999.000000,999.000000,0.000000,82.000000,4.018000e+03
max,6999.000000,4499.000000,90.000000,96.000000,5.287394e+06


In [11]:
# All records have type = game, confirming that the dataset contains only games
df["type"].value_counts(dropna=False)

type
game    985
Name: count, dtype: int64

In [12]:
# Info about developers and publishers
df[["developers", "publishers"]].head(5)

,developers,publishers
0,[Valve],[Valve]
1,[Mark Healey],[Mark Healey]
2,[Treyarch],[Activision]
3,[Sierra],[Activision]
4,[Sierra],[Activision]


In [13]:
df["developers"].apply(len).value_counts().sort_index()

developers
0      9
1    912
2     47
3     14
4      3
Name: count, dtype: int64

In [14]:
df["publishers"].apply(len).value_counts().sort_index()

# Developers and publishers are stored as lists because a game can have multiple.
# Most games have one developer and one publisher, but some have multiple or none.
# These fields should therefore be normalized into separate relational tables later.

publishers
0     68
1    847
2     44
3     24
4      2
Name: count, dtype: int64

In [15]:
# Info about genres and categories
df["genres"].apply(len).value_counts().sort_index()

genres
0      6
1    555
2    290
3    104
4     17
5      9
6      4
Name: count, dtype: int64

In [16]:
df["categories"].apply(len).value_counts().sort_index()

# Genres and categories are stored as lists of dictionaries, and games can have
# multiple genres or categories.
# The number of values varies between games, so these fields represent one-to-many relationships and should be normalized into separate tables later.

categories
0       1
1       9
2     334
3     217
4     100
5      66
6      54
7      41
8      32
9      38
10     22
11     15
12     11
13      8
14      9
15      5
16      2
17      4
18      2
19      2
20      3
21      5
24      1
25      1
27      1
32      1
34      1
Name: count, dtype: int64

In [17]:
df["genres"].explode().value_counts().head(20)
# id is string

genres
{'id': '1', 'description': 'Action'}                    393
{'id': '2', 'description': 'Strategy'}                  254
{'id': '4', 'description': 'Casual'}                    213
{'id': '23', 'description': 'Indie'}                    202
{'id': '25', 'description': 'Adventure'}                185
{'id': '3', 'description': 'RPG'}                       122
{'id': '28', 'description': 'Simulation'}               121
{'id': '9', 'description': 'Racing'}                     43
{'id': '37', 'description': 'Free To Play'}              20
{'id': '18', 'description': 'Sports'}                    19
{'id': '29', 'description': 'Massively Multiplayer'}     10
{'id': '72', 'description': 'Nudity'}                     1
{'id': '73', 'description': 'Violent'}                    1
Name: count, dtype: int64

In [18]:
df["categories"].explode().value_counts().head(20)
# id is int

categories
{'id': 62, 'description': 'Family Sharing'}                958
{'id': 2, 'description': 'Single-player'}                  944
{'id': 1, 'description': 'Multi-player'}                   396
{'id': 22, 'description': 'Steam Achievements'}            262
{'id': 23, 'description': 'Steam Cloud'}                   238
{'id': 18, 'description': 'Partial Controller Support'}    153
{'id': 9, 'description': 'Co-op'}                          139
{'id': 29, 'description': 'Steam Trading Cards'}           135
{'id': 25, 'description': 'Steam Leaderboards'}             96
{'id': 49, 'description': 'PvP'}                            83
{'id': 28, 'description': 'Full controller support'}        81
{'id': 44, 'description': 'Remote Play Together'}           67
{'id': 17, 'description': 'Includes level editor'}          64
{'id': 36, 'description': 'Online PvP'}                     62
{'id': 15, 'description': 'Stats'}                          57
{'id': 24, 'description': 'Shared/Split Scre

In [19]:
# Info about platforms
platform_columns = ["windows", "mac", "linux"]

print("Total games by platform:")
display(df[platform_columns].sum())

print("\nGames by platform combination:")
display(df[platform_columns].value_counts())

# Platform support is represented using boolean columns.
# All 985 games in this dataset support Windows, while 201 support Mac and 108 support Linux.
# Platform combinations show that most games support Windows only, while a smaller number also support Mac and/or Linux.

Total games by platform:


windows    985
mac        201
linux      108
dtype: int64


Games by platform combination:


windows  mac    linux
True     False  False    765
         True   False    112
                True      89
         False  True      19
Name: count, dtype: int64

In [20]:
# Info about release date
df["release_date"].value_counts().head(5)

release_date
30 Aug, 2006    15
                14
18 Jun, 2009    14
21 Apr, 2010    14
27 Aug, 2009     9
Name: count, dtype: int64

In [21]:
df["release_date"].head(5)

# Release dates are stored as strings using a consistent "DD Mon, YYYY" format.
# All records have a release date, and duplicate dates are expected because multiple games can be released on the same day.

0     1 Nov, 2000
1    12 Oct, 2005
2    18 Nov, 2008
3    23 Jul, 2009
4    23 Jul, 2009
Name: release_date, dtype: str

In [22]:
# Convert release_date from string to datetime for consistent date handling
df["release_date"] = pd.to_datetime(
    df["release_date"],
    format="%d %b, %Y"
)

df["release_date"].dtype

dtype('<M8[us]')

In [23]:
display(df["steam_appid"].duplicated().sum())
display(df["name"].duplicated().sum())

# No duplicate steam_appid or game names were found, appid can be used as the unique identifier for each game

np.int64(0)

np.int64(0)

In [24]:
# Unique values number
print("Unique developers:", len(set(
    developer
    for developers in df["developers"]
    for developer in developers
)))

print("Unique publishers:", len(set(
    publisher
    for publishers in df["publishers"]
    for publisher in publishers
)))

print("Unique genres:", len(set(
    genre["id"]
    for genres in df["genres"]
    for genre in genres
)))

print("Unique categories:", len(set(
    category["id"]
    for categories in df["categories"]
    for category in categories
)))

Unique developers: 516
Unique publishers: 251
Unique genres: 13
Unique categories: 58


In [25]:
# Asociate developers with steam id for better table representation
developers_df = df[["steam_appid", "developers"]].explode("developers")

developers_df.head(10)

,steam_appid,developers
0,10,Valve
1,1002,Mark Healey
2,10090,Treyarch
3,10100,Sierra
4,10110,Sierra
5,10120,Artech Studios
6,10130,Saber Interactive
7,10140,Wanako Games
8,10150,Radical Entertainment
9,10180,Infinity Ward


In [26]:
display(developers_df.shape)

display(developers_df["developers"].isna().sum())

# 1069 rows after explode
# 9 games without developers

(1069, 2)

np.int64(9)

In [27]:
# Remove rows without developers
developers_df = developers_df.dropna(subset=["developers"])
developers_df.shape

(1060, 2)

In [28]:
# Proceed the same for publishers
publishers_df = df[["steam_appid", "publishers"]].explode("publishers")

In [29]:
display(publishers_df.shape)
display(publishers_df["publishers"].isna().sum())

# 1083 rows after explode
# 68 games without publishers

(1083, 2)

np.int64(68)

In [30]:
publishers_df = publishers_df.dropna(subset=["publishers"])
publishers_df.shape

(1015, 2)

In [31]:
# Now clean genres
genres_df = df[["steam_appid", "genres"]].explode("genres")

display(genres_df.head(10))
display(genres_df.shape)
display(genres_df["genres"].isna().sum())

,steam_appid,genres
0,10,"{'id': '1', 'description': 'Action'}"
1,1002,"{'id': '23', 'description': 'Indie'}"
2,10090,"{'id': '1', 'description': 'Action'}"
3,10100,"{'id': '25', 'description': 'Adventure'}"
4,10110,"{'id': '25', 'description': 'Adventure'}"
5,10120,"{'id': '1', 'description': 'Action'}"
5,10120,"{'id': '4', 'description': 'Casual'}"
6,10130,"{'id': '1', 'description': 'Action'}"
7,10140,"{'id': '18', 'description': 'Sports'}"
8,10150,"{'id': '1', 'description': 'Action'}"


(1590, 2)

np.int64(6)

In [32]:
genres_df = genres_df.dropna(subset=["genres"])
genres_df.shape

(1584, 2)

In [33]:
# Clean categories, same as genres

categories_df = df[["steam_appid", "categories"]].explode("categories")

display(categories_df.head(10))
display(categories_df.shape)
display(categories_df["categories"].isna().sum())

,steam_appid,categories
0,10,"{'id': 1, 'description': 'Multi-player'}"
0,10,"{'id': 49, 'description': 'PvP'}"
0,10,"{'id': 36, 'description': 'Online PvP'}"
0,10,"{'id': 37, 'description': 'Shared/Split Screen..."
0,10,"{'id': 66, 'description': 'Color Alternatives'}"
0,10,"{'id': 68, 'description': 'Custom Volume Contr..."
0,10,"{'id': 75, 'description': 'Keyboard Only Option'}"
0,10,"{'id': 69, 'description': 'Stereo Sound'}"
0,10,"{'id': 8, 'description': 'Valve Anti-Cheat ena..."
0,10,"{'id': 62, 'description': 'Family Sharing'}"


(4571, 2)

np.int64(1)

In [34]:
categories_df = categories_df.dropna(subset=["categories"])
display(categories_df.shape)

(4570, 2)

In [35]:
# 4. Normalize fields ( genres, categories, producers, developers  )
# Normalize genres
genres_normalized = pd.json_normalize(genres_df["genres"])

display(genres_normalized.head())
display(genres_normalized.shape)

,id,description
0,1,Action
1,23,Indie
2,1,Action
3,25,Adventure
4,25,Adventure


(1584, 2)

In [36]:
# Reattach back the appid
genres_df = pd.concat(
    [
        genres_df[["steam_appid"]].reset_index(drop=True),
        genres_normalized.reset_index(drop=True)
    ],
    axis=1
)

display(genres_df.head(10))

,steam_appid,id,description
0,10,1,Action
1,1002,23,Indie
2,10090,1,Action
3,10100,25,Adventure
4,10110,25,Adventure
5,10120,1,Action
6,10120,4,Casual
7,10130,1,Action
8,10140,18,Sports
9,10150,1,Action


In [37]:
# Normalize Categories
categories_normalized = pd.json_normalize(categories_df["categories"])

display(categories_normalized.head())
display(categories_normalized.shape)

,id,description
0,1,Multi-player
0,49,PvP
0,36,Online PvP
0,37,Shared/Split Screen PvP
0,66,Color Alternatives


(4570, 2)

In [38]:
# Reattach back the appid
categories_df = pd.concat(
    [
        categories_df[["steam_appid"]].reset_index(drop=True),
        categories_normalized.reset_index(drop=True)
    ],
    axis=1
)

display(categories_df.head(10))

,steam_appid,id,description
0,10,1,Multi-player
1,10,49,PvP
2,10,36,Online PvP
3,10,37,Shared/Split Screen PvP
4,10,66,Color Alternatives
5,10,68,Custom Volume Controls
6,10,75,Keyboard Only Option
7,10,69,Stereo Sound
8,10,8,Valve Anti-Cheat enabled
9,10,62,Family Sharing


In [39]:
# Create a unique developer lookup table. I want to normalize the developer name to an unique id

developer_lookup = (
    developers_df[["developers"]]
    .drop_duplicates()
    .sort_values("developers")
    .reset_index(drop=True)
)

display(developer_lookup.head(10))
display(developer_lookup.shape)

,developers
0,.dat
1,14° East
2,1C: Maddox Games
3,1C:InoCo
4,2D BOY
5,2K Australia
6,2K Boston
7,2K China
8,2K Czech
9,2K Marin


(516, 1)

In [40]:
developer_lookup["developer_id"] = range(1, len(developer_lookup) + 1)

display(developer_lookup.head(10))

,developers,developer_id
0,.dat,1
1,14° East,2
2,1C: Maddox Games,3
3,1C:InoCo,4
4,2D BOY,5
5,2K Australia,6
6,2K Boston,7
7,2K China,8
8,2K Czech,9
9,2K Marin,10


In [41]:
game_developers = developers_df.merge(
    developer_lookup,
    on="developers",
    how="left"
)

display(game_developers.head(10))

,steam_appid,developers,developer_id
0,10,Valve,482
1,1002,Mark Healey,270
2,10090,Treyarch,462
3,10100,Sierra,396
4,10110,Sierra,396
5,10120,Artech Studios,30
6,10130,Saber Interactive,384
7,10140,Wanako Games,493
8,10150,Radical Entertainment,350
9,10180,Infinity Ward,227


In [42]:
game_developers = game_developers[
    ["steam_appid", "developer_id"]
]

display(game_developers.head(5))

display(game_developers["developer_id"].isna().sum())
display(game_developers.shape)

,steam_appid,developer_id
0,10,482
1,1002,270
2,10090,462
3,10100,396
4,10110,396


np.int64(0)

(1060, 2)

In [43]:
# Similarly, I will normalize the publishers

publisher_lookup = (
    publishers_df[["publishers"]]
    .drop_duplicates()
    .sort_values("publishers")
    .reset_index(drop=True)
)

display(publisher_lookup.head(5))
display(publisher_lookup.shape)

,publishers
0,1C-SoftClub
1,2D BOY
2,2K
3,3000AD
4,4sdk


(251, 1)

In [44]:
publisher_lookup["publisher_id"] = range(1, len(publisher_lookup) + 1)

display(publisher_lookup.head(10))

,publishers,publisher_id
0,1C-SoftClub,1
1,2D BOY,2
2,2K,3
3,3000AD,4
4,4sdk,5
5,ACE,6
6,ACE Team,7
7,Accelerate Games,8
8,Activision,9
9,Akella,10


In [45]:
game_publishers = publishers_df.merge(
    publisher_lookup,
    on="publishers",
    how="left"
)

display(game_publishers.head(10))

,steam_appid,publishers,publisher_id
0,10,Valve,230
1,1002,Mark Healey,137
2,10090,Activision,9
3,10100,Activision,9
4,10110,Activision,9
5,10120,Activision,9
6,10130,Activision,9
7,10140,Activision,9
8,10150,Activision,9
9,10180,Activision,9


In [46]:
game_publishers = game_publishers[
    ["steam_appid", "publisher_id"]
]

display(game_publishers.head(5))

display(game_publishers["publisher_id"].isna().sum())
display(game_publishers.shape)

,steam_appid,publisher_id
0,10,230
1,1002,137
2,10090,9
3,10100,9
4,10110,9


np.int64(0)

(1015, 2)

In [47]:
# In the same manner for genres
genre_lookup = (
    genres_df[["id", "description"]]
    .drop_duplicates()
    .sort_values("id")
    .reset_index(drop=True)
)

display(genre_lookup.head(10))
display(genre_lookup.shape)

,id,description
0,1,Action
1,18,Sports
2,2,Strategy
3,23,Indie
4,25,Adventure
5,28,Simulation
6,29,Massively Multiplayer
7,3,RPG
8,37,Free To Play
9,4,Casual


(13, 2)

In [48]:
# Convert genre id from string to int
display(genre_lookup["id"].dtype)

genre_lookup["id"] = pd.to_numeric(genre_lookup["id"])
display(genre_lookup.dtypes)
display(genre_lookup.head(10))

<StringDtype(na_value=nan)>

id             int64
description      str
dtype: object

,id,description
0,1,Action
1,18,Sports
2,2,Strategy
3,23,Indie
4,25,Adventure
5,28,Simulation
6,29,Massively Multiplayer
7,3,RPG
8,37,Free To Play
9,4,Casual


In [49]:
genres_df["id"] = pd.to_numeric(genres_df["id"])

game_genres = genres_df.merge(
    genre_lookup,
    on="id",
    how="left",
    suffixes=("_game", "_lookup")
)

display(game_genres.head(10))

,steam_appid,id,description_game,description_lookup
0,10,1,Action,Action
1,1002,23,Indie,Indie
2,10090,1,Action,Action
3,10100,25,Adventure,Adventure
4,10110,25,Adventure,Adventure
5,10120,1,Action,Action
6,10120,4,Casual,Casual
7,10130,1,Action,Action
8,10140,18,Sports,Sports
9,10150,1,Action,Action


In [50]:
game_genres = game_genres[
    ["steam_appid", "id"]
].rename(
    columns={"id": "genre_id"}
)

display(game_genres.head(10))


,steam_appid,genre_id
0,10,1
1,1002,23
2,10090,1
3,10100,25
4,10110,25
5,10120,1
6,10120,4
7,10130,1
8,10140,18
9,10150,1


In [51]:
display(game_genres["genre_id"].isna().sum())
display(game_genres.shape)

np.int64(0)

(1584, 2)

In [52]:
# Now for categories
category_lookup = (
    categories_df[["id", "description"]]
    .drop_duplicates()
    .sort_values("id")
    .reset_index(drop=True)
)

display(category_lookup.head(10))
display(category_lookup.shape)
display(category_lookup["id"].dtype)

,id,description
0,1,Multi-player
1,2,Single-player
2,6,Mods (require HL2)
3,8,Valve Anti-Cheat enabled
4,9,Co-op
5,13,Captions available
6,14,Commentary available
7,15,Stats
8,16,Includes Source SDK
9,17,Includes level editor


(58, 2)

dtype('int64')

In [53]:
game_categories = categories_df.merge(
    category_lookup,
    on="id",
    how="left",
    suffixes=("_game", "_lookup")
)

In [54]:
game_categories = game_categories[
    ["steam_appid", "id"]
].rename(
    columns={"id": "category_id"}
)

display(game_categories.head(10))

display(game_categories["category_id"].isna().sum())
display(game_categories.shape)

,steam_appid,category_id
0,10,1
1,10,49
2,10,36
3,10,37
4,10,66
5,10,68
6,10,75
7,10,69
8,10,8
9,10,62


np.int64(0)

(4570, 2)

In [55]:
# Final checks for developers/publishers

display(developer_lookup.shape)
display(developer_lookup["developer_id"].duplicated().sum())
display(game_developers["developer_id"].isna().sum())

display(publisher_lookup.shape)
display(publisher_lookup["publisher_id"].duplicated().sum())
display(game_publishers["publisher_id"].isna().sum())

(516, 2)

np.int64(0)

np.int64(0)

(251, 2)

np.int64(0)

np.int64(0)

In [56]:
# Final checks for genres/categories

display(genre_lookup.shape)
display(genre_lookup["id"].duplicated().sum())
display(game_genres["genre_id"].isna().sum())

display(category_lookup.shape)
display(category_lookup["id"].duplicated().sum())
display(game_categories["category_id"].isna().sum())

(13, 2)

np.int64(0)

np.int64(0)

(58, 2)

np.int64(0)

np.int64(0)

In [57]:
# 5. Create games DataFrame (all the other fields)
game_columns = [
    "steam_appid",
    "type",
    "name",
    "required_age",
    "is_free",
    "currency",
    "price_initial",
    "price_final",
    "discount_percent",
    "windows",
    "mac",
    "linux",
    "metacritic_score",
    "metacritic_url",
    "coming_soon",
    "release_date",
    "recommendations_total"
]

games_df = df[game_columns].copy()

display(games_df.head())
display(games_df.shape)

,steam_appid,type,name,required_age,is_free,currency,price_initial,price_final,discount_percent,windows,mac,linux,metacritic_score,metacritic_url,coming_soon,release_date,recommendations_total
0,10,game,Counter-Strike,0,False,EUR,819.0,819.0,0.0,True,True,True,88.0,https://www.metacritic.com/game/pc/counter-str...,False,2000-11-01,169887.0
1,1002,game,Rag Doll Kung Fu,0,False,EUR,79.0,79.0,0.0,True,False,False,69.0,https://www.metacritic.com/game/pc/rag-doll-ku...,False,2005-10-12,NaN
2,10090,game,Call of Duty: World at War,18,False,EUR,1999.0,1999.0,0.0,True,False,False,83.0,https://www.metacritic.com/game/pc/call-of-dut...,False,2008-11-18,45837.0
3,10100,game,King's Quest™ Collection,0,False,EUR,1999.0,1999.0,0.0,True,False,False,NaN,NaN,False,2009-07-23,331.0
4,10110,game,Space Quest™ Collection,0,False,EUR,999.0,999.0,0.0,True,False,False,NaN,NaN,False,2009-07-23,335.0


(985, 17)

In [58]:
display(games_df.info())

display(games_df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 985 entries, 0 to 984
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   steam_appid            985 non-null    int64         
 1   type                   985 non-null    str           
 2   name                   985 non-null    str           
 3   required_age           985 non-null    int64         
 4   is_free                985 non-null    bool          
 5   currency               925 non-null    str           
 6   price_initial          925 non-null    float64       
 7   price_final            925 non-null    float64       
 8   discount_percent       925 non-null    float64       
 9   windows                985 non-null    bool          
 10  mac                    985 non-null    bool          
 11  linux                  985 non-null    bool          
 12  metacritic_score       574 non-null    float64       
 13  metacritic_url  

None

steam_appid                0
type                       0
name                       0
required_age               0
is_free                    0
currency                  60
price_initial             60
price_final               60
discount_percent          60
windows                    0
mac                        0
linux                      0
metacritic_score         411
metacritic_url           411
coming_soon                0
release_date              14
recommendations_total    206
dtype: int64

In [59]:
# Some games have no release date provided by Steam.
# These values are kept as NaT rather than being filled artificially.
display(
    df[df["steam_appid"].isin(
        games_df.loc[games_df["release_date"].isna(), "steam_appid"]
    )][
        ["steam_appid", "name", "release_date"]
    ]
)

,steam_appid,name,release_date
26,11180,Sherlock Holmes: The Mystery of the Persian Ca...,NaT
233,22140,Penumbra: Requiem,NaT
433,33580,The Great Art Race,NaT
443,33750,Prison Tycoon 4: SuperMax,NaT
449,33900,Arma 2,NaT
506,36130,Tradewinds Caravans + Odyssey Pack,NaT
507,36150,Westward Collection,NaT
564,38480,Earthworm Jim,NaT
587,39550,SpellForce 2 - Anniversary Edition,NaT
764,48900,Saira,NaT


In [60]:
# Make sure negative values don't exist (min > 0)
# Make sure discount is under 100% ( discount max < 100)
display(
    games_df[
        ["price_initial", "price_final", "discount_percent"]
    ].describe()
)

,price_initial,price_final,discount_percent
count,925.000000,925.000000,925.000000
mean,1008.552432,883.574054,10.558919
std,631.177806,601.296521,26.338416
min,79.000000,79.000000,0.000000
25%,499.000000,499.000000,0.000000
50%,975.000000,819.000000,0.000000
75%,999.000000,999.000000,0.000000
max,6999.000000,4499.000000,90.000000


In [77]:

games_df["price_initial"] = games_df["price_initial"].astype("Int64")
games_df["price_final"] = games_df["price_final"].astype("Int64")
games_df["discount_percent"] = games_df["discount_percent"].astype("Int64")

display(games_df[["price_initial", "price_final", "discount_percent"]].dtypes)

price_initial       Int64
price_final         Int64
discount_percent    Int64
dtype: object

In [78]:
# 6. Export transformed data to Parquet
# Export the transformed relational tables to Parquet format.
# Parquet is used as a columnar storage format for the processed data before loading it into PostgreSQL.

# Save the games Dataframe
from pathlib import Path

OUTPUT_DIR = Path("../../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

games_df.to_parquet(
    OUTPUT_DIR / "games.parquet",
    index=False
)

In [94]:
# Final checks for games
games_check = pd.read_parquet(
    OUTPUT_DIR / "games.parquet"
)

display(games_check.shape)
display(games_check.head(10))

(985, 17)

,steam_appid,type,name,required_age,is_free,currency,price_initial,price_final,discount_percent,windows,mac,linux,metacritic_score,metacritic_url,coming_soon,release_date,recommendations_total
0,10,game,Counter-Strike,0,False,EUR,819,819,0,True,True,True,88.0,https://www.metacritic.com/game/pc/counter-str...,False,2000-11-01,169887.0
1,1002,game,Rag Doll Kung Fu,0,False,EUR,79,79,0,True,False,False,69.0,https://www.metacritic.com/game/pc/rag-doll-ku...,False,2005-10-12,NaN
2,10090,game,Call of Duty: World at War,18,False,EUR,1999,1999,0,True,False,False,83.0,https://www.metacritic.com/game/pc/call-of-dut...,False,2008-11-18,45837.0
3,10100,game,King's Quest™ Collection,0,False,EUR,1999,1999,0,True,False,False,NaN,NaN,False,2009-07-23,331.0
4,10110,game,Space Quest™ Collection,0,False,EUR,999,999,0,True,False,False,NaN,NaN,False,2009-07-23,335.0
5,10120,game,Aces of the Galaxy™,0,False,EUR,999,999,0,True,False,False,NaN,NaN,False,2009-07-23,NaN
6,10130,game,TimeShift™,0,False,EUR,1499,1499,0,True,False,False,71.0,https://www.metacritic.com/game/pc/timeshift?f...,False,2009-07-23,1133.0
7,10140,game,3D Ultra™ Minigolf Adventures,0,False,EUR,999,999,0,True,False,False,NaN,NaN,False,2009-07-23,117.0
8,10150,game,Prototype™,0,False,EUR,1999,1999,0,True,False,False,79.0,https://www.metacritic.com/game/pc/prototype?f...,False,2009-06-12,12558.0
9,10180,game,Call of Duty®: Modern Warfare® 2 (2009),0,False,EUR,1999,1999,0,True,True,False,86.0,https://www.metacritic.com/game/pc/call-of-dut...,False,2009-11-11,35854.0


In [95]:
# Save the developers Dataframe

developers = developer_lookup.rename(
    columns={
        "developers": "name"
    }
).copy()

display(developers.head())
display(developers.shape)


,name,developer_id
0,.dat,1
1,14° East,2
2,1C: Maddox Games,3
3,1C:InoCo,4
4,2D BOY,5


(516, 2)

In [96]:
developers.to_parquet(
    OUTPUT_DIR / "developers.parquet",
    index=False
)

In [97]:
game_developers_processed = game_developers.copy()

display(game_developers_processed.head())
display(game_developers_processed.shape)


,steam_appid,developer_id
0,10,482
1,1002,270
2,10090,462
3,10100,396
4,10110,396


(1060, 2)

In [98]:
game_developers_processed.to_parquet(
    OUTPUT_DIR / "game_developers.parquet",
    index=False
)

In [99]:
# Final checks for developers
developers_check = pd.read_parquet(
    OUTPUT_DIR / "developers.parquet"
)

game_developers_check = pd.read_parquet(
    OUTPUT_DIR / "game_developers.parquet"
)

display(developers_check.shape)
display(game_developers_check.shape)


(516, 2)

(1060, 2)

In [100]:
# Export publishers 

publishers = publisher_lookup.rename(
    columns={"publishers": "name"}
).copy()

publishers.to_parquet(
    OUTPUT_DIR / "publishers.parquet",
    index=False
)

game_publishers.to_parquet(
    OUTPUT_DIR / "game_publishers.parquet",
    index=False
)

In [101]:
# Export genres

genres = genre_lookup.rename(
    columns={"id": "genre_id"}
).copy()

genres.to_parquet(
    OUTPUT_DIR / "genres.parquet",
    index=False
)

game_genres.to_parquet(
    OUTPUT_DIR / "game_genres.parquet",
    index=False
)

In [102]:
# Export categories

categories = category_lookup.rename(
    columns={"id": "category_id"}
).copy()

categories.to_parquet(
    OUTPUT_DIR / "categories.parquet",
    index=False
)

game_categories.to_parquet(
    OUTPUT_DIR / "game_categories.parquet",
    index=False
)

In [103]:
# 7. Final final checks 

# Verify that all processed Parquet files were created successfully and contain the expected data structures.

processed_files = [
    "games.parquet",
    "developers.parquet",
    "game_developers.parquet",
    "publishers.parquet",
    "game_publishers.parquet",
    "genres.parquet",
    "game_genres.parquet",
    "categories.parquet",
    "game_categories.parquet",
]

for file_name in processed_files:
    path = OUTPUT_DIR / file_name
    print(f"{file_name}: {'OK' if path.exists() else 'MISSING'}")

games.parquet: OK
developers.parquet: OK
game_developers.parquet: OK
publishers.parquet: OK
game_publishers.parquet: OK
genres.parquet: OK
game_genres.parquet: OK
categories.parquet: OK
game_categories.parquet: OK


In [104]:
for file_name in processed_files:
    path = OUTPUT_DIR / file_name
    table = pd.read_parquet(path)
    print(f"{file_name}: {table.shape}")

games.parquet: (985, 17)
developers.parquet: (516, 2)
game_developers.parquet: (1060, 2)
publishers.parquet: (251, 2)
game_publishers.parquet: (1015, 2)
genres.parquet: (13, 2)
game_genres.parquet: (1584, 2)
categories.parquet: (58, 2)
game_categories.parquet: (4570, 2)


In [105]:
for file_name in processed_files:
    table = pd.read_parquet(OUTPUT_DIR / file_name)
    print(f"\n--- {file_name} ---")
    print(table.columns.tolist())


--- games.parquet ---
['steam_appid', 'type', 'name', 'required_age', 'is_free', 'currency', 'price_initial', 'price_final', 'discount_percent', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'coming_soon', 'release_date', 'recommendations_total']

--- developers.parquet ---
['name', 'developer_id']

--- game_developers.parquet ---
['steam_appid', 'developer_id']

--- publishers.parquet ---
['name', 'publisher_id']

--- game_publishers.parquet ---
['steam_appid', 'publisher_id']

--- genres.parquet ---
['genre_id', 'description']

--- game_genres.parquet ---
['steam_appid', 'genre_id']

--- categories.parquet ---
['category_id', 'description']

--- game_categories.parquet ---
['steam_appid', 'category_id']
